# Random forest hyperparameters
The four forest-level settings on the concentric-circles data, then checks of `max_features`, `bootstrap`,
`warm_start` and `ccp_alpha`. The interactive version is `app.py` (`python app.py`, then open
http://127.0.0.1:8050).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

## 1. The data: 500 points, 2 columns, 2 classes; 375 for training

In [ ]:
df = pd.read_csv("data/concertriccir2.csv")
X, y = df.iloc[:, :2].values, df.iloc[:, -1].values.astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)   # 25% test by default; used by the figures
print(X_train.shape, X_test.shape)

# One split of 125 test points is noisy (one point = 0.008), so the scores below average 20 random splits.
splits = [train_test_split(X, y, random_state=s) for s in range(20)]


def score(**settings):
    """Mean test accuracy of a random forest with these settings over the 20 splits."""
    accs = [RandomForestClassifier(random_state=s, n_jobs=-1, **settings).fit(a, c).score(b, d)
            for s, (a, b, c, d) in enumerate(splits)]
    return round(float(np.mean(accs)), 3)

## 2. n_estimators: the number of trees

In [ ]:
for n in [1, 5, 10, 50, 100, 200]:
    print(f"{n:4d} trees: {score(n_estimators=n)}")

## 3. max_samples: rows per tree (out of 375), with 50 trees

In [ ]:
for rows in [25, 50, 100, 200, 280, None]:
    print(f"{str(rows):>4s} rows: {score(n_estimators=100, max_samples=rows)}")   # None = all 375 rows

## 4. max_features: columns considered at each split, with 50 trees

In [ ]:
for mf in ["sqrt", 1, 2]:
    print(f"max_features={mf!r}: {score(n_estimators=100, max_features=mf)}")

In [ ]:
# How many columns do "sqrt", "log2" and a decimal mean? max_features_ of a fitted tree shows the number.
X100, y100 = make_classification(n_samples=300, n_features=100, random_state=0)
for mf in ["sqrt", "log2", 0.2, None]:
    rf = RandomForestClassifier(n_estimators=3, max_features=mf, random_state=0).fit(X100, y100)
    print(f"100 columns, max_features={mf!r}: {rf.estimators_[0].max_features_} columns per split")

In [ ]:
# "auto" was removed in scikit-learn 1.3
try:
    RandomForestClassifier(max_features="auto").fit(X_train, y_train)
except Exception as e:
    print(type(e).__name__, ":", str(e)[:150])

## 5. bootstrap
`bootstrap=False` gives every tree the whole training set; `max_samples` is then not allowed.

In [ ]:
print("bootstrap=True :", score(n_estimators=100))
print("bootstrap=False:", score(n_estimators=100, bootstrap=False))
try:
    score(n_estimators=100, bootstrap=False, max_samples=0.5)
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# bootstrap=False and every column at every split: almost no randomness is left, so the trees are nearly the same
# (they differ only where two splits tie and the column order, shuffled at random, breaks the tie)
rf = RandomForestClassifier(n_estimators=10, bootstrap=False, max_features=None, random_state=42).fit(X_train, y_train)
print("leaves per tree:", [t.get_n_leaves() for t in rf.estimators_])

## 6. warm_start: add trees to a trained forest

In [ ]:
rf = RandomForestClassifier(n_estimators=50, warm_start=True, random_state=42).fit(X_train, y_train)
print(len(rf.estimators_), "trees, accuracy", rf.score(X_test, y_test))
rf.set_params(n_estimators=100)     # ask for 100 trees ...
rf.fit(X_train, y_train)            # ... only the 50 new trees are trained
print(len(rf.estimators_), "trees, accuracy", rf.score(X_test, y_test))

## 7. ccp_alpha: cost-complexity pruning of every tree

In [ ]:
for a in [0.0, 0.002, 0.01, 0.05]:
    rf = RandomForestClassifier(n_estimators=100, ccp_alpha=a, random_state=42, n_jobs=-1).fit(X_train, y_train)
    leaves = np.mean([t.get_n_leaves() for t in rf.estimators_])
    print(f"ccp_alpha={a}: {leaves:5.1f} leaves per tree on average, mean test accuracy {score(n_estimators=100, ccp_alpha=a):.3f}")

## 8. The regressor: same settings, different criteria and a different max_features default

In [ ]:
print("classifier criteria:", sorted(RandomForestClassifier._parameter_constraints["criterion"][0].options))
print("regressor criteria: ", sorted(RandomForestRegressor._parameter_constraints["criterion"][0].options))
print("max_features default: classifier", RandomForestClassifier().max_features,
      "| regressor", RandomForestRegressor().max_features)